## Zrób to sam. Polecane przećwiczyć przed kolejnymi ćwiczeniami -- klasyfikacja win. TODO

Spróbujmy przewidzieć ocenę wina na podstawie jego parametrów

In [2]:
import pandas as pd
import torch
df = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv', delimiter=";")
original_df = df
print(df)

      fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0               7.0              0.27         0.36            20.7      0.045   
1               6.3              0.30         0.34             1.6      0.049   
2               8.1              0.28         0.40             6.9      0.050   
3               7.2              0.23         0.32             8.5      0.058   
4               7.2              0.23         0.32             8.5      0.058   
...             ...               ...          ...             ...        ...   
4893            6.2              0.21         0.29             1.6      0.039   
4894            6.6              0.32         0.36             8.0      0.047   
4895            6.5              0.24         0.19             1.2      0.041   
4896            5.5              0.29         0.30             1.1      0.022   
4897            6.0              0.21         0.38             0.8      0.020   

      free sulfur dioxide  

#### ... Jakieś wstępne przetwarzanie danych?

In [4]:
train=df.sample(frac=0.8,random_state=200) #random state is a seed value
test=df.drop(train.index)
print(f'{train.shape=}')
print(f'{test.shape=}')

train.shape=(3918, 12)
test.shape=(980, 12)


### Standaryzacja danych

In [216]:
from sklearn.preprocessing import StandardScaler

X_train = train.values[:, :-1]
y_train = train.values[:, -1]

X_test = test.values[:, :-1]
y_test = test.values[:, -1]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit + transform
X_test_scaled = scaler.transform(X_test)  # tylko transform

In [217]:
import torch.utils.data as data
import torch.nn as nn
import numpy as np
import torch.nn.functional as F

In [221]:
class SimpleClassifier(nn.Module):

    def __init__(self, num_inputs, num_hidden, num_outputs):
        super().__init__()
        # Initialize the modules we need to build the network
        self.linear1 = nn.Linear(num_inputs, num_hidden)
        self.act_fn = nn.ReLU()
        self.linear2 = nn.Linear(num_hidden, num_outputs)

        nn.init.xavier_uniform_(self.linear1.weight)
        nn.init.zeros_(self.linear1.bias)
        nn.init.xavier_uniform_(self.linear2.weight)
        nn.init.zeros_(self.linear2.bias)

    #         self.linear3 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Perform the calculation of the model to determine the prediction
        x = self.linear1(x)
        x = self.act_fn(x)
        x = self.linear2(x)
        #         x = self.act_fn(x)
        #         x = self.linear3(x)
        return x

In [222]:
def trained_model(model_class, output_size,hidden_size, batch_size, loss_module, optimizer, lr, num_epochs, verbose=0):
    train_dataset = data.TensorDataset(
        torch.from_numpy(X_train_scaled).float(),
        torch.from_numpy(y_train).long()
    )
    train_data_loader = data.DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True
    )

    model = model_class(
        num_inputs=next(iter(train_data_loader))[0].shape[1],
        num_outputs=output_size,
        # num_outputs=next(iter(train_data_loader))[1].shape[1] if next(iter(train_data_loader))[1].ndim > 1 else 1,
        num_hidden=hidden_size,
    )
    optimizer = optimizer(model.parameters(), lr=lr)

    model.train()  # switches to train mode

    for epoch in range(num_epochs):
        for data_input, data_label in train_data_loader:
            # print(data_input)
            pred = model(data_input)
            # print(pred)
            # print(pred.shape)
            # pred = pred.squeeze(dim=1)
            # print(pred.shape)
            # print(data_label)
            # print(data_label.shape)
            # print(pred.shape)
            # print(data_label.shape)

            loss = loss_module(pred, data_label)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        if verbose > 0:
            print(f"Epoch: {epoch}, loss: {loss.item():.3}")

    return model

def test_model(model, output_size, batch_size):
    test_dataset = data.TensorDataset(
        torch.from_numpy(X_test_scaled).float(),
        torch.from_numpy(y_test).float()
    )
    test_data_loader = data.DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False, drop_last=False
    )

    model.eval()  # Set model to eval mode
    true_preds, num_preds = 0.0, 0.0

    with torch.no_grad():  # Deactivate gradients for the following code
        for data_inputs, data_labels in test_data_loader:
            # Determine prediction of model on dev set
            # data_inputs, data_labels = data_inputs.to(device), data_labels.to(device)
            preds = model(data_inputs)
            print(preds.min().item(), preds.max().item())
            # print(preds.shape)
            # preds = preds.squeeze(dim=1)
            # print(preds.shape)
            pred_class = torch.argmax(preds, dim=1)  # Softmax to choose max
            # print(pred_class.shape)
            # print(data_labels.shape)
            # true_class = torch.argmax(data_labels, dim=1)
            # print(true_class.shape)

            # Keep records of predictions for the accuracy metric (true_preds=TP+TN, num_preds=TP+TN+FP+FN)
            true_preds += (pred_class == data_labels).sum()
            num_preds += data_labels.shape[0]

    acc = true_preds / num_preds
    print(f"Accuracy of the model: {100.0*acc:4.2f}%")

In [223]:
BATCH_SIZE = 64
LEARNING_RATE_OPTIMIZER = 0.001
OPTIMIZER = torch.optim.Adam
LOSS_MODULE = nn.CrossEntropyLoss()
NUM_EPOCHS = 200
OUTPUT_SIZE = 10
HIDDEN_SIZE = 64

model = trained_model(model_class=SimpleClassifier, output_size=OUTPUT_SIZE, hidden_size=HIDDEN_SIZE, batch_size=BATCH_SIZE, loss_module=LOSS_MODULE, optimizer=OPTIMIZER, lr=LEARNING_RATE_OPTIMIZER, num_epochs=NUM_EPOCHS, verbose=1)
test_model(model, output_size=OUTPUT_SIZE, batch_size=BATCH_SIZE)


Epoch: 0, loss: 1.37
Epoch: 1, loss: 1.11
Epoch: 2, loss: 1.12
Epoch: 3, loss: 1.1
Epoch: 4, loss: 0.884
Epoch: 5, loss: 1.23
Epoch: 6, loss: 1.11
Epoch: 7, loss: 0.853
Epoch: 8, loss: 0.93
Epoch: 9, loss: 0.98
Epoch: 10, loss: 1.27
Epoch: 11, loss: 1.11
Epoch: 12, loss: 0.617
Epoch: 13, loss: 1.3
Epoch: 14, loss: 0.831
Epoch: 15, loss: 1.33
Epoch: 16, loss: 1.01
Epoch: 17, loss: 0.8
Epoch: 18, loss: 1.23
Epoch: 19, loss: 0.956
Epoch: 20, loss: 0.706
Epoch: 21, loss: 1.45
Epoch: 22, loss: 0.77
Epoch: 23, loss: 0.759
Epoch: 24, loss: 1.36
Epoch: 25, loss: 1.02
Epoch: 26, loss: 1.26
Epoch: 27, loss: 1.09
Epoch: 28, loss: 1.05
Epoch: 29, loss: 0.921
Epoch: 30, loss: 0.794
Epoch: 31, loss: 0.804
Epoch: 32, loss: 0.982
Epoch: 33, loss: 1.09
Epoch: 34, loss: 1.59
Epoch: 35, loss: 0.978
Epoch: 36, loss: 0.593
Epoch: 37, loss: 1.02
Epoch: 38, loss: 1.07
Epoch: 39, loss: 1.14
Epoch: 40, loss: 0.611
Epoch: 41, loss: 0.722
Epoch: 42, loss: 0.843
Epoch: 43, loss: 0.644
Epoch: 44, loss: 1.13
Epoch: